# WildDex — SageMaker Training
**EfficientNetB0 transfer learning · 500 species · TFLite export**

### Workflow
1. Upload local `dataset/` to S3
2. Launch SageMaker training job (GPU instance)
3. Download model artifact → convert to TFLite
4. Integrate into app

### Prerequisites
- AWS account with SageMaker access
- Run in **SageMaker Studio** or locally with `pip install sagemaker boto3`
- Dataset already downloaded: `scripts/download_images.py` puts images in `dataset/train/` and `dataset/val/`

In [ ]:
# ── 1. Install / import ──────────────────────────────────────────────────────
!pip install -q sagemaker boto3 tensorflow

import os, json, tarfile, subprocess
import boto3
import sagemaker
from sagemaker import get_execution_role
from sagemaker.tensorflow import TensorFlow
from sagemaker.inputs import TrainingInput
from pathlib import Path

print('SageMaker SDK:', sagemaker.__version__)

In [ ]:
# ── 2. AWS session & role ────────────────────────────────────────────────────
# If running locally, set your profile: boto3.setup_default_session(profile_name='YOUR_PROFILE')
session    = sagemaker.Session()
bucket     = session.default_bucket()          # auto-creates sagemaker-<region>-<account>
prefix     = 'wilddex-500'
region     = session.boto_region_name

# In SageMaker Studio: role = get_execution_role()
# Running locally: paste your SageMaker execution role ARN here
role = get_execution_role()  # or: 'arn:aws:iam::123456789:role/SageMakerExecutionRole'

print(f'Region : {region}')
print(f'Bucket : {bucket}')
print(f'Prefix : {prefix}')
print(f'Role   : {role[:60]}...')

In [ ]:
# ── 3. Upload dataset to S3 ──────────────────────────────────────────────────
# This uploads dataset/train/ and dataset/val/ to s3://<bucket>/wilddex-500/
# Skips existing files — safe to re-run if interrupted.

DATASET_DIR = '../dataset'   # relative to scripts/ — adjust if running from repo root
s3 = boto3.client('s3')

def upload_folder(local_dir: str, s3_prefix: str):
    local = Path(local_dir)
    files = list(local.rglob('*.jpg')) + list(local.rglob('*.jpeg')) + list(local.rglob('*.png'))
    print(f'Uploading {len(files):,} images from {local_dir}...')
    for i, f in enumerate(files):
        key = f'{s3_prefix}/{f.relative_to(local)}'
        s3.upload_file(str(f), bucket, key)
        if (i + 1) % 500 == 0:
            print(f'  {i+1:,}/{len(files):,} uploaded...')
    print(f'  Done. {len(files):,} files → s3://{bucket}/{s3_prefix}/')

upload_folder(f'{DATASET_DIR}/train', f'{prefix}/train')
upload_folder(f'{DATASET_DIR}/val',   f'{prefix}/val')

s3_train = f's3://{bucket}/{prefix}/train'
s3_val   = f's3://{bucket}/{prefix}/val'
print(f'\nS3 train: {s3_train}')
print(f'S3 val  : {s3_val}')

In [ ]:
# ── 4. Write the training script ─────────────────────────────────────────────
# SageMaker runs this script inside a managed TensorFlow container.
# It reads data from env vars SM_CHANNEL_TRAIN / SM_CHANNEL_VAL,
# and saves the model to SM_MODEL_DIR.

train_script = '''
import os, argparse, json
import tensorflow as tf
from tensorflow.keras import layers

def parse_args():
    p = argparse.ArgumentParser()
    p.add_argument('--epochs-frozen',   type=int, default=15)
    p.add_argument('--epochs-finetune', type=int, default=20)
    p.add_argument('--batch-size',      type=int, default=32)
    p.add_argument('--lr-head',         type=float, default=1e-3)
    p.add_argument('--lr-finetune',     type=float, default=1e-5)
    p.add_argument('--model-dir',       type=str, default=os.environ.get('SM_MODEL_DIR', '/tmp/model'))
    p.add_argument('--train',           type=str, default=os.environ.get('SM_CHANNEL_TRAIN', 'dataset/train'))
    p.add_argument('--val',             type=str, default=os.environ.get('SM_CHANNEL_VAL',   'dataset/val'))
    return p.parse_args()

def main():
    args = parse_args()
    IMAGE_SIZE = (224, 224)

    # ── Data ────────────────────────────────────────────────────────────────
    train_ds = tf.keras.preprocessing.image_dataset_from_directory(
        args.train, image_size=IMAGE_SIZE, batch_size=args.batch_size,
        label_mode='int', seed=42,
    )
    val_ds = tf.keras.preprocessing.image_dataset_from_directory(
        args.val, image_size=IMAGE_SIZE, batch_size=args.batch_size,
        label_mode='int', seed=42,
    )
    class_names = train_ds.class_names
    num_classes = len(class_names)
    print(f"Classes: {num_classes}")

    AUTOTUNE = tf.data.AUTOTUNE
    train_ds = train_ds.cache().shuffle(2000).prefetch(AUTOTUNE)
    val_ds   = val_ds.cache().prefetch(AUTOTUNE)

    # ── Augmentation ────────────────────────────────────────────────────────
    aug = tf.keras.Sequential([
        layers.RandomFlip("horizontal"),
        layers.RandomRotation(0.15),
        layers.RandomZoom(0.15),
        layers.RandomBrightness(0.15),
        layers.RandomContrast(0.15),
    ], name="augmentation")

    # ── Model ────────────────────────────────────────────────────────────────
    base = tf.keras.applications.EfficientNetB0(
        input_shape=(224, 224, 3), include_top=False, weights="imagenet"
    )
    base.trainable = False

    inputs  = tf.keras.Input(shape=(224, 224, 3))
    x       = aug(inputs)
    x       = base(x, training=False)
    x       = layers.GlobalAveragePooling2D()(x)
    x       = layers.BatchNormalization()(x)
    x       = layers.Dropout(0.3)(x)
    outputs = layers.Dense(num_classes, activation="softmax")(x)
    model   = tf.keras.Model(inputs, outputs)

    # ── Phase 1: Train head ──────────────────────────────────────────────────
    model.compile(
        optimizer=tf.keras.optimizers.Adam(args.lr_head),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    callbacks1 = [
        tf.keras.callbacks.EarlyStopping(patience=4, restore_best_weights=True),
        tf.keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=2, verbose=1),
    ]
    print("=== Phase 1: Training head ===")
    model.fit(train_ds, validation_data=val_ds, epochs=args.epochs_frozen, callbacks=callbacks1)

    # ── Phase 2: Fine-tune top 30 base layers ─────────────────────────────
    base.trainable = True
    for layer in base.layers[:-30]:
        layer.trainable = False

    model.compile(
        optimizer=tf.keras.optimizers.Adam(args.lr_finetune),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    callbacks2 = [
        tf.keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True),
        tf.keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=3, verbose=1),
    ]
    print("=== Phase 2: Fine-tuning ===")
    model.fit(train_ds, validation_data=val_ds, epochs=args.epochs_finetune, callbacks=callbacks2)

    # ── Save ─────────────────────────────────────────────────────────────────
    os.makedirs(args.model_dir, exist_ok=True)
    model.save(os.path.join(args.model_dir, "wilddex_model.keras"))

    with open(os.path.join(args.model_dir, "labels.json"), "w") as f:
        json.dump(class_names, f)

    val_loss, val_acc = model.evaluate(val_ds)
    print(f"\nFinal val accuracy: {val_acc*100:.1f}%")
    print("Training complete.")

if __name__ == "__main__":
    main()
'''

Path('wilddex_train_script.py').write_text(train_script)
print('Training script written: wilddex_train_script.py')

In [ ]:
# ── 5. Configure & launch SageMaker training job ────────────────────────────
# Instance options (cost per hour, ~35-50min training time):
#   ml.p2.xlarge  — K80 GPU  — ~$0.90/hr  (cheapest, fine for 500 classes)
#   ml.p3.2xlarge — V100 GPU — ~$3.06/hr  (3x faster)
#   ml.g4dn.xlarge— T4 GPU   — ~$0.74/hr  (good price/perf)

estimator = TensorFlow(
    entry_point='wilddex_train_script.py',
    role=role,
    instance_count=1,
    instance_type='ml.g4dn.xlarge',     # T4 GPU — best value
    framework_version='2.13',
    py_version='py310',
    output_path=f's3://{bucket}/{prefix}/output',
    hyperparameters={
        'epochs-frozen':   15,
        'epochs-finetune': 20,
        'batch-size':      32,
        'lr-head':         1e-3,
        'lr-finetune':     1e-5,
    },
    base_job_name='wilddex-500species',
)

estimator.fit(
    inputs={
        'train': TrainingInput(s3_train, content_type='application/x-image'),
        'val':   TrainingInput(s3_val,   content_type='application/x-image'),
    },
    wait=True,   # blocks until done — set False to run async
    logs='All',
)

print('\nJob complete.')
print('Model artifact:', estimator.model_data)

In [ ]:
# ── 6. Download model artifact from S3 ──────────────────────────────────────
import tarfile

model_tar_path = 'wilddex_model.tar.gz'

# estimator.model_data = 's3://<bucket>/<prefix>/output/<job>/output/model.tar.gz'
s3_uri = estimator.model_data
bucket_name, key = s3_uri.replace('s3://', '').split('/', 1)

s3.download_file(bucket_name, key, model_tar_path)
print(f'Downloaded: {model_tar_path}')

with tarfile.open(model_tar_path) as tar:
    tar.extractall('model_output')

print('Extracted to model_output/')
import os
for f in os.listdir('model_output'):
    print(' ', f)

In [ ]:
# ── 7. Convert to TFLite (float16 quantization) ──────────────────────────────
import tensorflow as tf, json

model = tf.keras.models.load_model('model_output/wilddex_model.keras')
print('Model loaded. Output shape:', model.output_shape)

converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.target_spec.supported_types = [tf.float16]
tflite_model = converter.convert()

with open('wilddex_model.tflite', 'wb') as f:
    f.write(tflite_model)

size_mb = len(tflite_model) / 1024 / 1024
print(f'Saved wilddex_model.tflite — {size_mb:.1f} MB')

# Save labels
with open('model_output/labels.json') as f:
    labels = json.load(f)
with open('labels.txt', 'w') as f:
    f.write('\n'.join(labels))
print(f'Saved labels.txt — {len(labels)} classes')

In [ ]:
# ── 8. Quick sanity test ─────────────────────────────────────────────────────
import numpy as np

interpreter = tf.lite.Interpreter(model_path='wilddex_model.tflite')
interpreter.allocate_tensors()

input_details  = interpreter.get_input_details()
output_details = interpreter.get_output_details()

print('Input  shape:', input_details[0]['shape'])   # should be [1, 224, 224, 3]
print('Output shape:', output_details[0]['shape'])  # should be [1, 500]

# Feed a random image — check it produces valid probabilities
dummy = np.random.rand(1, 224, 224, 3).astype(np.float32)
interpreter.set_tensor(input_details[0]['index'], dummy)
interpreter.invoke()
probs = interpreter.get_tensor(output_details[0]['index'])[0]

top5_idx = np.argsort(probs)[-5:][::-1]
print('\nTop-5 predictions (random input — expect ~0.2% each):')
for i in top5_idx:
    print(f'  {labels[i]:<30} {probs[i]*100:.3f}%')
print('\nModel ready for app integration.')

In [ ]:
# ── 9. Copy to app (update paths as needed) ──────────────────────────────────
import shutil

APP_ASSETS = '../assets'  # adjust to your app's assets folder

shutil.copy('wilddex_model.tflite', f'{APP_ASSETS}/wilddex_model.tflite')
shutil.copy('labels.txt',           f'{APP_ASSETS}/labels.txt')

print('Copied to app assets.')
print('Next steps:')
print('  1. Install react-native-fast-tflite in the app')
print('  2. Load model on CameraScreen')
print('  3. Run inference before calling Sonnet (hybrid approach)')
print('  4. Fall back to Sonnet if TFLite confidence < 0.7 or species not in model')

## Architecture notes (for the interview)

### End-to-end pipeline

| Step | Tool | Why |
|------|------|-----|
| Data collection | iNaturalist API | Research-grade community photos, labeled by experts, free |
| Base model | EfficientNetB0 | Best accuracy/size tradeoff for mobile — 4M params, ~29MB |
| Framework | Keras (via TensorFlow 2.x) | High-level API, clean model definition, SavedModel/TFLite export |
| Training infra | SageMaker `TensorFlow` Estimator | Managed GPU instances, S3 I/O, auto artifact storage, reproducible |
| Quantization | TFLite float16 | Halves model size (~14MB), <1% accuracy drop vs float32 |
| On-device inference | react-native-fast-tflite | Runs on-device via NNAPI/Metal — zero latency, no API cost |
| Fallback | Claude Sonnet | Catches species outside the 500-class model |

---

### Model architecture (Keras Functional API)

```
Input (224×224×3)
  │
  ▼
Data Augmentation          ← RandomFlip, Rotation±15%, Zoom±15%, Brightness, Contrast
  │                           Applied only at training time — improves generalization on small datasets
  ▼
EfficientNetB0 (frozen)    ← Pretrained on ImageNet (1.28M images, 1000 classes)
  │                           Uses MBConv blocks: depthwise separable convolutions + squeeze-excitation
  │                           Compound scaling balances depth × width × resolution together
  ▼
GlobalAveragePooling2D     ← Collapses 7×7×1280 spatial feature map → (1280,)
  │                           Avoids flattening bloat; spatial-invariant summary of features
  ▼
BatchNormalization         ← Normalizes activations before dropout; stabilizes and speeds convergence
  │
  ▼
Dropout(0.3)               ← Randomly zeros 30% of neurons during training — reduces overfitting
  │
  ▼
Dense(500, softmax)        ← Output: probability distribution over 500 species classes
```

---

### Two-phase training strategy

**Why two phases?** Pretrained ImageNet weights are valuable — we don't want a high LR to destroy them before the new head is calibrated.

**Phase 1 — Head only (base frozen, up to 15 epochs)**
- All 237 EfficientNetB0 layers are frozen (`base.trainable = False`)
- Only the 4 new layers (GAP → BN → Dropout → Dense) are trained
- LR = `1e-3` (Adam) — high enough to converge quickly
- EarlyStopping patience=4 stops when val_loss plateaus
- Goal: get the classification head to ~50–60% val accuracy before unfreezing

**Phase 2 — Fine-tune top 30 base layers (up to 20 epochs)**
- `base.trainable = True`, but `base.layers[:-30]` are re-frozen
- Only the last 30 layers of EfficientNetB0 update (high-level semantic detectors)
- LR drops to `1e-5` — tiny steps prevent catastrophic forgetting of ImageNet features
- ReduceLROnPlateau halves LR on plateau; EarlyStopping patience=5
- Goal: adapt high-level features from generic ImageNet → wildlife photography domain

**Why 30 layers?** The first ~207 layers learn universal low-level features (edges, textures, gradients) that transfer perfectly to any image domain. The last 30 learn high-level semantics (object parts, species-specific patterns) that benefit from domain adaptation.

---

### Callbacks

| Callback | Config | Effect |
|----------|--------|--------|
| `EarlyStopping` | patience=4 / 5 | Stops training early if val_loss stalls; restores best weights automatically |
| `ReduceLROnPlateau` | factor=0.5, patience=2/3 | Halves LR when val_loss plateaus — helps escape shallow local minima |

---

### Data decisions

| Decision | Reason |
|----------|--------|
| 50 images/class | Minimum viable for transfer learning; pretrained features do the heavy lifting |
| Research-grade filter | iNaturalist community-verified IDs — reduces label noise significantly |
| `order_by=votes` | Fetches most-upvoted (highest quality) observations first |
| Separate `train/` and `val/` dirs | Clean 80/20 split at download time; no leakage risk |
| iNaturalist source | 150M+ observations, expert-verified, globally distributed species coverage |

---

### SageMaker mechanics

- **`TensorFlow` Estimator** — wraps the training script in a managed TF container; handles Docker, EC2, CloudWatch logs automatically
- **Training channels** — `SM_CHANNEL_TRAIN` / `SM_CHANNEL_VAL` env vars point to S3 paths, mounted at `/opt/ml/input/data/<channel>/` inside the container
- **Model artifacts** — training script saves to `SM_MODEL_DIR` (`/opt/ml/model/`); SageMaker tars everything → uploads to S3 automatically at job end
- **Hyperparameter injection** — passed as CLI args to the script; trivially swappable for SageMaker Hyperparameter Tuning (HPO) jobs
- **Instance choice** — `ml.g4dn.xlarge` has a T4 GPU (16GB VRAM), 4 vCPUs, ~$0.74/hr; full job costs ~$0.50

---

### Expected results

| Images/class | Expected top-1 val accuracy |
|-------------|----------------------------|
| 50 | 70–80% |
| 100 | 80–87% |
| 200 | 87–92% |

App confidence threshold = **0.7** — predictions below this fall back to Claude Sonnet, which handles rare/ambiguous species and breeds not in the 500-class model.

**Training time:** ~35–50 min on `ml.g4dn.xlarge`  
**Final model size:** ~14 MB (float16 TFLite)